## Setup & Imports

In [ ]:
# Setup path for imports
import sys
import os
import json
from pathlib import Path

# Ensure we're in the backend directory for imports
backend_dir = Path.cwd()
if backend_dir.name != 'backend':
    backend_dir = backend_dir / 'backend' if (backend_dir / 'backend').exists() else backend_dir
sys.path.insert(0, str(backend_dir))
os.chdir(backend_dir)

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes backend: {'backend' in str(sys.path)}")

In [ ]:
# Helper function to pretty print JSON results
def pp(result):
    """Pretty print JSON string or dict"""
    if isinstance(result, str):
        try:
            data = json.loads(result)
            print(json.dumps(data, indent=2))
        except:
            print(result)
    else:
        print(json.dumps(result, indent=2, default=str))

# Helper to truncate long content
def truncate(text, max_len=500):
    return text[:max_len] + '...' if len(text) > max_len else text

---
## 1. Tavily Search Tools (NEW)

Testing the new Tavily integration for web search with date filtering.

In [ ]:
# Import Tavily tools
from app.mcp.tools.tavily_search import web_search, news_search, extract_content, crawl_website

print("Tavily tools loaded:")
print(f"  - web_search: {web_search.name}")
print(f"  - news_search: {news_search.name}")
print(f"  - extract_content: {extract_content.name}")
print(f"  - crawl_website: {crawl_website.name}")

### 1.1 Web Search - Recent (time_range)

In [ ]:
# Test web_search with time_range filter (last week)
result = web_search.invoke({
    "query": "carbon fiber composites aerospace news",
    "max_results": 5,
    "time_range": "week"
})
pp(result)

### 1.2 Web Search - Historical Date Range (Quarterly Recap)

In [ ]:
# Test web_search with specific date range (Q3 2025: Jul-Sep)
result = web_search.invoke({
    "query": "Hexcel Toray carbon fiber competitive landscape",
    "max_results": 5,
    "start_date": "2025-07-01",
    "end_date": "2025-09-30"
})
pp(result)

### 1.3 News Search - Recent News

In [ ]:
# Test news_search (optimized for news articles)
result = news_search.invoke({
    "query": "advanced materials supply chain",
    "max_results": 5,
    "time_range": "week"
})
pp(result)

### 1.4 Extract Content - Product Page

In [ ]:
# Test extract_content (for pulling content from specific URLs)
# Example: Hexcel product page
result = extract_content.invoke({
    "urls": ["https://www.hexcel.com/Products/Carbon-Fiber"]
})
pp(result)

### 1.5 Crawl Website - Find Documents

In [ ]:
# Test crawl_website (discover pages on a domain)
result = crawl_website.invoke({
    "url": "https://www.hexcel.com/Products",
    "search_query": "prepreg datasheet",
    "max_pages": 5
})
pp(result)

---
## 2. Industry Watcher Tools

News, press releases, and social intelligence.

In [ ]:
# Import Industry Watcher tools
from app.mcp.tools.fmp_news import get_general_news, get_stock_news, get_press_releases, search_press_releases
from app.mcp.tools.x_social import search_x_posts, get_x_user_posts

print("Industry Watcher tools loaded")

### 2.1 FMP General News

In [ ]:
# Test general news
result = get_general_news.invoke({"limit": 5})
pp(result)

### 2.2 Stock-Specific News

In [ ]:
# Test stock news for a tracked company
result = get_stock_news.invoke({"symbol": "HXL", "limit": 5})  # Hexcel
pp(result)

### 2.3 Press Releases

In [ ]:
# Test press releases
result = get_press_releases.invoke({"symbol": "HXL", "limit": 5})
pp(result)

### 2.4 X/Twitter Search (Requires X_BEARER_TOKEN)

In [ ]:
# Test X search (if token configured)
result = search_x_posts.invoke({
    "query": "carbon fiber aerospace",
    "max_results": 10
})
pp(result)

---
## 3. Market Scout Tools

Financial data, earnings transcripts, commodities.

In [ ]:
# Import Market Scout tools
from app.mcp.tools.api_ninja_earnings import get_earnings_transcript_ninja, search_earnings_keywords
from app.mcp.tools.fmp_financial import get_earnings_transcript_latest, get_commodities_list

print("Market Scout tools loaded")

### 3.1 Earnings Transcript (API Ninjas - Primary)

In [ ]:
# Test earnings transcript from API Ninjas
result = get_earnings_transcript_ninja.invoke({"ticker": "HXL"})
# Truncate the long transcript
data = json.loads(result)
if 'transcript' in data:
    data['transcript'] = truncate(data['transcript'], 1000)
pp(data)

### 3.2 Search Earnings Keywords

In [ ]:
# Search for specific keywords in earnings transcripts
result = search_earnings_keywords.invoke({
    "ticker": "HXL",
    "keywords": ["carbon fiber", "aerospace", "supply chain", "capacity"]
})
pp(result)

### 3.3 Commodities List

In [ ]:
# Get commodities data
result = get_commodities_list.invoke({})
pp(result)

---
## 4. Product Maven Tools

Product intelligence, document extraction, competitive analysis.

In [ ]:
# Import Product Maven tools
from app.mcp.tools.product_intelligence import (
    search_competitor_products, get_product_specs, compare_product_specs,
    find_product_gaps, get_c1_catalog, suggest_alternatives
)
from app.mcp.tools.document_extraction import crawl_for_documents, extract_pdf_content

print("Product Maven tools loaded")

### 4.1 Search Competitor Products

In [ ]:
# Search for competitor products
result = search_competitor_products.invoke({
    "query": "carbon fiber prepreg",
    "company": "Hexcel"
})
pp(result)

### 4.2 Get C1 Catalog

In [ ]:
# Get C1's product catalog
result = get_c1_catalog.invoke({"category": "prepreg"})
pp(result)

### 4.3 Crawl for Documents (Custom - finds PDF links)

In [ ]:
# Find PDF documents on a page
result = crawl_for_documents.invoke({
    "url": "https://www.hexcel.com/Products/Carbon-Fiber",
    "file_types": ["pdf"],
    "filter_keywords": ["datasheet", "tds", "technical"]
})
pp(result)

### 4.4 Extract PDF Content

In [ ]:
# Extract content from a PDF (use a URL from crawl_for_documents)
# Example - replace with actual PDF URL
# result = extract_pdf_content.invoke({
#     "url": "https://example.com/datasheet.pdf",
#     "pages": [1, 2]
# })
# pp(result)
print("Uncomment and provide a PDF URL to test")

---
## 5. Corpus Query Tools (ChromaDB)

Query the pre-ingested intelligence corpus.

In [ ]:
# Import Corpus tools
from app.mcp.tools.corpus_query import (
    query_corpus, get_digest, get_entity_mentions,
    get_corpus_stats, get_priority_items, get_competitor_activity, get_market_signals
)

print("Corpus query tools loaded")

### 5.1 Corpus Stats

In [ ]:
# Get corpus statistics
result = get_corpus_stats.invoke({})
pp(result)

### 5.2 Query Corpus

In [ ]:
# Semantic search in corpus
result = query_corpus.invoke({
    "query": "carbon fiber supply chain challenges",
    "n_results": 5
})
pp(result)

### 5.3 Get Entity Mentions

In [ ]:
# Get mentions of a specific entity
result = get_entity_mentions.invoke({"entity": "Hexcel", "n_results": 5})
pp(result)

### 5.4 Get Daily Digest

In [ ]:
# Get digest of recent intelligence
result = get_digest.invoke({"days": 7, "limit": 10})
pp(result)

### 5.5 Competitor Activity

In [ ]:
# Get competitor activity
result = get_competitor_activity.invoke({"company": "Toray", "n_results": 5})
pp(result)

---
## 6. Data Management Tools

User preferences and tracking lists.

In [ ]:
# Import Data Management tools
from app.mcp.tools.data_management import (
    get_companies_list, get_materials_list, get_preferences
)

print("Data Management tools loaded")

### 6.1 Get Tracked Companies

In [ ]:
# Get list of tracked companies
result = get_companies_list.invoke({})
pp(result)

### 6.2 Get Tracked Materials

In [ ]:
# Get list of tracked materials
result = get_materials_list.invoke({})
pp(result)

### 6.3 Get User Preferences

In [ ]:
# Get user preferences
result = get_preferences.invoke({})
pp(result)

---
## 7. Full Agent Test

Test the complete agent workflow via API.

In [ ]:
import requests

BASE_URL = "http://localhost:8000"

def chat(message: str, mode: str = "pulse") -> dict:
    """Send a chat message to the API"""
    response = requests.post(
        f"{BASE_URL}/api/chat",
        json={"message": message, "mode": mode}
    )
    return response.json()

print("Chat helper ready. Modes: pulse, industry, market, products, data")

### 7.1 Industry Watcher Agent

In [ ]:
# Test Industry Watcher
result = chat("What's the latest news on carbon fiber aerospace?", mode="industry")
print("Response:")
print(result.get('message', {}).get('content', 'No response')[:2000])

### 7.2 Market Scout Agent

In [ ]:
# Test Market Scout
result = chat("What did Hexcel say about supply chain in their latest earnings?", mode="market")
print("Response:")
print(result.get('message', {}).get('content', 'No response')[:2000])

### 7.3 Product Maven Agent

In [ ]:
# Test Product Maven
result = chat("What carbon fiber prepreg products does Hexcel offer?", mode="products")
print("Response:")
print(result.get('message', {}).get('content', 'No response')[:2000])

### 7.4 Pulse Mode (Router)

In [ ]:
# Test Pulse mode (auto-routing)
result = chat("Give me a quick update on what's happening with our tracked companies", mode="pulse")
print("Response:")
print(result.get('message', {}).get('content', 'No response')[:2000])

---
## Notes & Observations

Use this section to document findings during eval.

In [ ]:
# Notes:
# - 
# - 
# - 